# FotMob — Premier League 2024/25 Player Stats Scraper
**Pipeline:** Fixtures → Scrape match pages → Parse player stats → Save to Parquet

## Cell 1 — Imports & configuration

In [ ]:
import hashlib
import hmac
import time
import random
import json
import asyncio
import logging
import requests
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('fotmob')

# FotMob config
BASE_URL    = 'https://www.fotmob.com/api'
LEAGUE_ID   = 47
SEASON      = '2024/2025'
HMAC_SECRET = 'cZ7pzSrpDV2lJfCYqDfmBMBt3FfWREFx'

# Rate limiting
MAX_CONCURRENT = 4
JITTER_MIN     = 1.0
JITTER_MAX     = 3.0
MAX_RETRIES    = 5
BACKOFF_BASE   = 2

# Storage
RAW_DIR = Path('./data/raw')
OUT_DIR = Path('./data/output')
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# FotMob stat category title → clean snake_case name
CATEGORY_MAP = {
    'Top stats'       : 'top_stats',
    'Attack'          : 'attack',
    'Defence'         : 'defense',
    'Defense'         : 'defense',
    'Passes'          : 'passes',
    'Duels'           : 'duels',
    'Goalkeeping'     : 'goalkeeping',
    'Physical metrics': 'physical_metrics',
}

# Goalkeeper metric keys used to reclassify their single Top stats block
GK_GOALKEEPING = {
    'saves', 'goals_conceded', 'xgot_faced', 'goals_prevented',
    'diving_save', 'saves_inside_box', 'acted_as_sweeper',
    'punches', 'throws', 'high_claim',
}
GK_PASSING = {
    'accurate_passes', 'accurate_long_balls',
    'passes_into_final_third', 'accurate_crosses',
}
GK_DEFENSE = {
    'recoveries', 'clearances', 'defensive_actions',
    'tackles', 'interceptions', 'shot_blocks',
}
GK_DUELS = {
    'ground_duels_won', 'aerials_won',
    'was_fouled', 'fouls_committed', 'aerial_duels_won',
}

print('✓ Setup complete')

✓ Setup complete


c:\Users\natmaw\anaconda3\envs\mlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — Auth headers & async HTTP client

In [2]:
def make_headers(path: str) -> dict:
    """Build x-mas HMAC-signed headers for FotMob API requests."""
    epoch_ms = str(int(time.time() * 1000))
    digest   = hmac.new(
        HMAC_SECRET.encode(),
        f"{epoch_ms}/{path.lstrip('/')}".encode(),
        hashlib.sha1,
    ).hexdigest()
    return {
        'x-mas'          : f'{epoch_ms}.{digest}',
        'User-Agent'     : 'Mozilla/5.0 (iPhone; CPU iPhone OS 17_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
        'Accept'         : 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate',
        'Referer'        : 'https://www.fotmob.com/',
        'Origin'         : 'https://www.fotmob.com',
    }


# Headers for scraping match HTML pages
PAGE_HEADERS = {
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept'         : 'text/html,application/xhtml+xml,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate',
    'Referer'        : 'https://www.fotmob.com/',
}


class FotMobClient:
    """Async HTTP client — injects auth headers, rate limits, retries on 429/5xx."""

    def __init__(self):
        self._session   = None
        self._semaphore = asyncio.Semaphore(MAX_CONCURRENT)

    async def __aenter__(self):
        self._session = aiohttp.ClientSession(
            timeout=aiohttp.ClientTimeout(total=30),
            connector=aiohttp.TCPConnector(ssl=True, limit=MAX_CONCURRENT),
        )
        return self

    async def __aexit__(self, *_):
        if self._session:
            await self._session.close()

    async def get(self, path: str, params: dict = None) -> dict:
        url = f'{BASE_URL}{path}'
        for attempt in range(1, MAX_RETRIES + 1):
            async with self._semaphore:
                await asyncio.sleep(random.uniform(JITTER_MIN, JITTER_MAX))
                try:
                    async with self._session.get(
                        url, params=params, headers=make_headers(path)
                    ) as resp:
                        if resp.status == 200:
                            return await resp.json(content_type=None)
                        elif resp.status in (429, 503):
                            await asyncio.sleep(BACKOFF_BASE ** attempt + random.random())
                        elif resp.status >= 500:
                            await asyncio.sleep(BACKOFF_BASE ** attempt)
                        else:
                            raise ValueError(f'HTTP {resp.status} on {url}')
                except aiohttp.ClientConnectorError:
                    await asyncio.sleep(BACKOFF_BASE ** attempt)
        raise RuntimeError(f'Exhausted {MAX_RETRIES} retries for {url}')


print('✓ Headers + client defined')

✓ Headers + client defined


## Cell 3 — Fetch all fixtures (38 matchdays)

In [3]:
async def fetch_all_fixtures() -> pd.DataFrame:
    """Single API call returns all 380 fixtures for the full season."""
    async with FotMobClient() as client:
        data = await client.get(
            '/leagues',
            params={'id': LEAGUE_ID, 'ccode3': 'GBR', 'season': SEASON},
        )

    with open(RAW_DIR / 'league_meta.json', 'w') as f:
        json.dump(data, f, indent=2)

    all_matches = data.get('fixtures', {}).get('allMatches', [])
    if not all_matches:
        print('⚠ No matches found — check league_meta.json')
        return pd.DataFrame()

    rows = []
    for m in all_matches:
        rows.append({
            'match_id'  : m.get('id'),
            'round'     : m.get('round'),
            'page_url'  : m.get('pageUrl', ''),
            'match_date': m.get('status', {}).get('utcTime', ''),
            'finished'  : m.get('status', {}).get('finished', False),
            'home_team' : m.get('home', {}).get('name', ''),
            'home_id'   : m.get('home', {}).get('id'),
            'away_team' : m.get('away', {}).get('name', ''),
            'away_id'   : m.get('away', {}).get('id'),
        })

    df = pd.DataFrame(rows).dropna(subset=['match_id'])
    df['match_id'] = df['match_id'].astype(int)
    df = df.sort_values(['round', 'match_date']).reset_index(drop=True)
    df.to_parquet(OUT_DIR / 'fixtures.parquet', index=False)
    df.to_csv(OUT_DIR / 'fixtures.csv', index=False)
    log.info('Saved %d fixtures', len(df))
    return df


try:
    fixtures_df = await fetch_all_fixtures()
except RuntimeError:
    fixtures_df = asyncio.run(fetch_all_fixtures())

print(f'Total    : {len(fixtures_df)}')
print(f'Finished : {fixtures_df["finished"].sum()}')
print(f'Unplayed : {(~fixtures_df["finished"]).sum()}')
fixtures_df.head()

00:44:55  INFO      Saved 380 fixtures


Total    : 380
Finished : 380
Unplayed : 0


,match_id,round,page_url,match_date,finished,home_team,home_id,away_team,away_id
0,4506263,1,/matches/fulham-vs-manchester-united/3cqww9#45...,2024-08-16T19:00:00Z,True,Manchester United,10260,Fulham,9879
1,4506264,1,/matches/liverpool-vs-ipswich-town/2ugv0q#4506264,2024-08-17T11:30:00Z,True,Ipswich Town,9902,Liverpool,8650
2,4506265,1,/matches/wolverhampton-wanderers-vs-arsenal/2t...,2024-08-17T14:00:00Z,True,Arsenal,9825,Wolverhampton Wanderers,8602
3,4506266,1,/matches/everton-vs-brighton-hove-albion/2y16f...,2024-08-17T14:00:00Z,True,Everton,8668,Brighton & Hove Albion,10204
4,4506267,1,/matches/southampton-vs-newcastle-united/2weqv...,2024-08-17T14:00:00Z,True,Newcastle United,10261,Southampton,8466


## Cell 4 — Match page scraper

In [4]:
def fetch_match_page(page_url: str) -> dict | None:
    """
    Fetch a match page and extract the __NEXT_DATA__ JSON blob.
    Player stats are embedded server-side — no JS execution needed.
    """
    full_url = f'https://www.fotmob.com{page_url}'
    try:
        resp = requests.get(full_url, headers=PAGE_HEADERS, timeout=20)
        if resp.status_code != 200:
            log.warning('HTTP %s for %s', resp.status_code, full_url)
            return None
        tag = BeautifulSoup(resp.text, 'html.parser').find('script', {'id': '__NEXT_DATA__'})
        if not tag:
            log.warning('No __NEXT_DATA__ at %s', full_url)
            return None
        return json.loads(tag.string)
    except Exception as exc:
        log.warning('Error fetching %s: %s', full_url, exc)
        return None


print('✓ fetch_match_page() defined')

✓ fetch_match_page() defined


## Cell 5 — Player stats parser

In [5]:
def parse_match_page(next_data: dict, match_row: pd.Series) -> list[dict]:
    """
    Parse __NEXT_DATA__ into flat stat rows.

    Output: one row per (player x match x category x metric)

    Stat structure in playerStats:
      - Outfield players: 4 blocks (Top stats / Attack / Defense / Duels)
      - Goalkeepers     : 1 giant Top stats block — reclassified by metric key

    Each block's 'stats' field is a dict:
      { "Metric Name": { "key": "metric_key", "stat": { "value": 3, "type": "integer" } } }
    """
    rows         = []
    page_props   = next_data.get('props', {}).get('pageProps', {})
    content      = page_props.get('content', {})
    player_stats = content.get('playerStats', {})

    if not player_stats:
        return rows

    match_ctx = {
        'match_id'  : match_row['match_id'],
        'round'     : match_row['round'],
        'match_date': match_row['match_date'],
        'home_team' : match_row['home_team'],
        'away_team' : match_row['away_team'],
    }

    for pid_str, pdata in player_stats.items():
        if not isinstance(pdata, dict):
            continue

        is_gk = pdata.get('isGoalkeeper', False)

        player_ctx = {
            'player_id'    : pdata.get('id', pid_str),
            'player_name'  : pdata.get('name', ''),
            'team_id'      : pdata.get('teamId'),
            'team_name'    : pdata.get('teamName', ''),
            'is_goalkeeper': is_gk,
            'shirt_number' : pdata.get('shirtNumber'),
            'position_id'  : pdata.get('positionId'),
        }

        for block in pdata.get('stats', []):
            if not isinstance(block, dict):
                continue

            raw_title  = block.get('title', '')
            block_key  = block.get('key', '')
            stats_dict = block.get('stats', {})

            if not isinstance(stats_dict, dict):
                continue

            for metric_name, metric_data in stats_dict.items():
                if not isinstance(metric_data, dict):
                    continue

                metric_key = metric_data.get('key', '')
                stat       = metric_data.get('stat', {})
                value      = stat.get('value')
                total      = stat.get('total')     # denominator for fractions e.g. 3/5
                stat_type  = stat.get('type', '')

                # Reclassify goalkeeper metrics out of the single Top stats block
                if is_gk and block_key == 'top_stats':
                    if metric_key in GK_GOALKEEPING:
                        category = 'goalkeeping'
                    elif metric_key in GK_PASSING:
                        category = 'passes'
                    elif metric_key in GK_DEFENSE:
                        category = 'defense'
                    elif metric_key in GK_DUELS:
                        category = 'duels'
                    else:
                        category = 'top_stats'
                else:
                    category = CATEGORY_MAP.get(raw_title, block_key)

                rows.append({
                    **match_ctx,
                    **player_ctx,
                    'category'  : category,
                    'metric'    : metric_name,
                    'metric_key': metric_key,
                    'value'     : value,
                    'total'     : total,
                    'stat_type' : stat_type,
                })

    return rows


print('✓ parse_match_page() defined')

✓ parse_match_page() defined


## Cell 6 — Full season scrape

In [6]:
def scrape_full_season(fixtures_df: pd.DataFrame) -> pd.DataFrame:
    """
    Scrape and parse all finished matches.

    - Loads from data/raw/<match_id>.json if already cached
    - Only hits the network for uncached matches
    - Silently skips matches where playerStats is None (unplayed)
    - Fully resumable — re-run after any interruption
    """
    finished = fixtures_df[fixtures_df['finished']].copy().reset_index(drop=True)
    all_rows = []
    failed   = []

    already_cached = sum(
        1 for _, r in finished.iterrows()
        if (RAW_DIR / (str(r['match_id']) + '.json')).exists()
    )
    print(f'Matches to scrape : {len(finished)}')
    print(f'Already cached    : {already_cached}')
    print(f'New requests      : {len(finished) - already_cached}')
    print()

    for _, match_row in tqdm(finished.iterrows(), total=len(finished), desc='Scraping season', unit='match'):
        match_id = match_row['match_id']
        raw_path = RAW_DIR / f'{match_id}.json'

        if raw_path.exists():
            with open(raw_path) as f:
                next_data = json.load(f)
        else:
            time.sleep(random.uniform(JITTER_MIN, JITTER_MAX))
            next_data = fetch_match_page(match_row['page_url'])
            if next_data is None:
                failed.append(match_id)
                continue
            with open(raw_path, 'w') as f:
                json.dump(next_data, f)

        # Skip silently if playerStats is None — match not played yet
        content      = next_data.get('props', {}).get('pageProps', {}).get('content', {})
        player_stats = content.get('playerStats')
        if not player_stats:
            continue

        rows = parse_match_page(next_data, match_row)
        if rows:
            all_rows.extend(rows)
        else:
            failed.append(match_id)

    df = pd.DataFrame(all_rows)

    if not df.empty:
        df['match_id']   = df['match_id'].astype(int)
        df['round']      = df['round'].astype(int)
        df['value']      = pd.to_numeric(df['value'], errors='coerce')
        df['total']      = pd.to_numeric(df['total'], errors='coerce')
        df['match_date'] = pd.to_datetime(df['match_date'], utc=True, errors='coerce')

    print(f'\n{"="*50}')
    print(f'  Total stat rows  : {len(df):,}')
    print(f'  Matches parsed   : {df["match_id"].nunique() if not df.empty else 0}')
    print(f'  Unique players   : {df["player_id"].nunique() if not df.empty else 0}')
    print(f'  Categories       : {sorted(df["category"].unique().tolist()) if not df.empty else []}')
    print(f'  Failed           : {len(failed)}')
    print(f'{"="*50}')
    return df


stats_df = scrape_full_season(fixtures_df)
stats_df.head()

Matches to scrape : 380
Already cached    : 0
New requests      : 380



Scraping season: 100%|██████████| 380/380 [17:46<00:00,  2.81s/match]



  Total stat rows  : 255,658
  Matches parsed   : 262
  Unique players   : 610
  Categories       : ['attack', 'defense', 'duels', 'goalkeeping', 'passes', 'physical_metrics', 'top_stats']
  Failed           : 0


,match_id,round,match_date,home_team,away_team,player_id,player_name,team_id,team_name,is_goalkeeper,shirt_number,position_id,category,metric,metric_key,value,total,stat_type
0,4506263,1,2024-08-16 19:00:00+00:00,Manchester United,Fulham,182962,Tom Cairney,9879,Fulham,False,10,NaN,top_stats,FotMob rating,rating_title,6.22,NaN,double
1,4506263,1,2024-08-16 19:00:00+00:00,Manchester United,Fulham,182962,Tom Cairney,9879,Fulham,False,10,NaN,top_stats,Minutes played,minutes_played,11.00,NaN,integer
2,4506263,1,2024-08-16 19:00:00+00:00,Manchester United,Fulham,182962,Tom Cairney,9879,Fulham,False,10,NaN,top_stats,Goals,goals,0.00,NaN,integer
3,4506263,1,2024-08-16 19:00:00+00:00,Manchester United,Fulham,182962,Tom Cairney,9879,Fulham,False,10,NaN,top_stats,Assists,assists,0.00,NaN,integer
4,4506263,1,2024-08-16 19:00:00+00:00,Manchester United,Fulham,182962,Tom Cairney,9879,Fulham,False,10,NaN,top_stats,Expected assists (xA),expected_assists,0.01,NaN,double


## Cell 7 — Save to Parquet + CSV

In [7]:
stats_df.to_parquet(OUT_DIR / 'player_stats.parquet', index=False, engine='pyarrow')
stats_df.to_csv(OUT_DIR / 'player_stats.csv', index=False)

parquet_mb = (OUT_DIR / 'player_stats.parquet').stat().st_size / 1e6
csv_mb     = (OUT_DIR / 'player_stats.csv').stat().st_size / 1e6

print(f'✓ player_stats.parquet  — {parquet_mb:.1f} MB')
print(f'✓ player_stats.csv      — {csv_mb:.1f} MB')
print(f'  Rows       : {len(stats_df):,}')
print(f'  Matches    : {stats_df["match_id"].nunique()}')
print(f'  Players    : {stats_df["player_id"].nunique()}')
print(f'  Categories : {sorted(stats_df["category"].unique().tolist())}')
print()
print('Rows per category:')
print(stats_df.groupby('category').size().sort_values(ascending=False).to_string())

✓ player_stats.parquet  — 0.8 MB
✓ player_stats.csv      — 44.3 MB
  Rows       : 255,658
  Matches    : 262
  Players    : 610
  Categories : ['attack', 'defense', 'duels', 'goalkeeping', 'passes', 'physical_metrics', 'top_stats']

Rows per category:
category
top_stats           104596
defense              57526
attack               46328
duels                42830
goalkeeping           2630
passes                1028
physical_metrics       720


## Cell 8 — Reload & reparse from cache (re-entry point)

In [8]:
def reparse_from_cache(fixtures_df: pd.DataFrame) -> pd.DataFrame:
    """
    Reparse all cached raw JSON files without hitting the network.
    Use this when you update the parser and want to reprocess existing data.
    """
    finished = fixtures_df[fixtures_df['finished']].copy().reset_index(drop=True)
    cached   = [
        row for _, row in finished.iterrows()
        if (RAW_DIR / f"{row['match_id']}.json").exists()
    ]
    all_rows = []
    failed   = []

    print(f'Cached files to reparse: {len(cached)}')

    for match_row in tqdm(cached, desc='Reparsing', unit='match'):
        match_id = match_row['match_id']
        try:
            with open(RAW_DIR / f'{match_id}.json') as f:
                next_data = json.load(f)

            content      = next_data.get('props', {}).get('pageProps', {}).get('content', {})
            player_stats = content.get('playerStats')
            if not player_stats:
                continue

            rows = parse_match_page(next_data, match_row)
            if rows:
                all_rows.extend(rows)
            else:
                failed.append(match_id)
        except Exception as exc:
            log.warning('Failed reparsing %s: %s', match_id, exc)
            failed.append(match_id)

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df['match_id']   = df['match_id'].astype(int)
        df['round']      = df['round'].astype(int)
        df['value']      = pd.to_numeric(df['value'], errors='coerce')
        df['total']      = pd.to_numeric(df['total'], errors='coerce')
        df['match_date'] = pd.to_datetime(df['match_date'], utc=True, errors='coerce')

    print(f'\n{"="*50}')
    print(f'  Total stat rows  : {len(df):,}')
    print(f'  Matches parsed   : {df["match_id"].nunique() if not df.empty else 0}')
    print(f'  Unique players   : {df["player_id"].nunique() if not df.empty else 0}')
    print(f'  Categories       : {sorted(df["category"].unique().tolist()) if not df.empty else []}')
    print(f'  Failed           : {len(failed)}')
    print(f'{"="*50}')
    return df


# Uncomment to reparse without re-scraping:
# stats_df = reparse_from_cache(fixtures_df)

# Uncomment to reload saved parquet directly:
# stats_df = pd.read_parquet(OUT_DIR / 'player_stats.parquet')

print('✓ reparse_from_cache() defined')

✓ reparse_from_cache() defined


## Cell 9 — Sanity check queries

In [9]:
# Top rated players (min 5 appearances)
top_rated = (
    stats_df[['match_id', 'player_id', 'player_name', 'team_name', 'metric_key', 'value']]
    .query('metric_key == "rating_title"')
    .groupby(['player_id', 'player_name', 'team_name'])
    .agg(appearances=('match_id', 'count'), avg_rating=('value', 'mean'))
    .query('appearances >= 5')
    .sort_values('avg_rating', ascending=False)
    .head(10)
    .reset_index()
)
print('── Top 10 rated players (min 5 appearances) ──')
print(top_rated[['player_name', 'team_name', 'appearances', 'avg_rating']].to_string(index=False))
print()

# Goals leaderboard
goals = (
    stats_df
    .query('metric_key == "goals"')
    .groupby(['player_id', 'player_name', 'team_name'])['value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
goals.columns = ['player_id', 'player_name', 'team_name', 'goals']
print('── Top 10 scorers ──')
print(goals[['player_name', 'team_name', 'goals']].to_string(index=False))
print()

# Top goalkeepers by saves
saves = (
    stats_df
    .query('metric_key == "saves"')
    .groupby(['player_id', 'player_name', 'team_name'])['value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
saves.columns = ['player_id', 'player_name', 'team_name', 'saves']
print('── Top 10 goalkeepers by saves ──')
print(saves[['player_name', 'team_name', 'saves']].to_string(index=False))
print()

# Pass accuracy leaders (min 5 appearances)
passes = (
    stats_df
    .query('metric_key == "accurate_passes" and total > 0')
    .assign(pct=lambda x: x['value'] / x['total'] * 100)
    .groupby(['player_id', 'player_name', 'team_name'])
    .agg(appearances=('match_id', 'count'), avg_pass_pct=('pct', 'mean'))
    .query('appearances >= 5')
    .sort_values('avg_pass_pct', ascending=False)
    .head(10)
    .reset_index()
)
print('── Top 10 pass accuracy (min 5 appearances) ──')
print(passes[['player_name', 'team_name', 'appearances', 'avg_pass_pct']].to_string(index=False))

── Top 10 rated players (min 5 appearances) ──
       player_name               team_name  appearances  avg_rating
      Bryan Mbeumo               Brentford            6    8.643333
   Bruno Fernandes       Manchester United           26    8.263846
      Mark Flekken               Brentford            6    8.213333
    Alexander Isak        Newcastle United            6    7.950000
Dominik Szoboszlai               Liverpool           18    7.880000
       Yoane Wissa               Brentford            6    7.880000
     Omar Marmoush         Manchester City           10    7.852000
   Ibrahima Konaté               Liverpool           12    7.843333
   Rayan Ait Nouri Wolverhampton Wanderers            6    7.823333
      Levi Colwill                 Chelsea            6    7.786667

── Top 10 scorers ──
       player_name                team_name  goals
      Jarrod Bowen          West Ham United   12.0
    Benjamin Sesko        Manchester United   12.0
       Cole Palmer            

In [10]:
# Load from parquet if not already in memory
# stats_df = pd.read_parquet(OUT_DIR / 'player_stats.parquet')

# ── All unique metrics grouped by category and player type ────────
summary = (
    stats_df
    .groupby(['is_goalkeeper', 'category', 'metric_key', 'metric'])
    .agg(
        appearances = ('value', 'count'),
        non_null    = ('value', lambda x: x.notna().sum()),
        sample_val  = ('value', 'mean'),
    )
    .reset_index()
)

summary['player_type'] = summary['is_goalkeeper'].map({True: 'goalkeeper', False: 'outfield'})
summary = summary.drop(columns='is_goalkeeper').sort_values(['player_type', 'category', 'metric_key'])

print(f'Total unique metrics : {summary["metric_key"].nunique()}')
print(f'Outfield metrics     : {summary[summary["player_type"] == "outfield"]["metric_key"].nunique()}')
print(f'Goalkeeper metrics   : {summary[summary["player_type"] == "goalkeeper"]["metric_key"].nunique()}')
print()

# ── Print outfield metrics by category ────────────────────────────
print('=' * 70)
print('OUTFIELD PLAYER METRICS')
print('=' * 70)
for cat, group in summary[summary['player_type'] == 'outfield'].groupby('category'):
    print(f'\n  [{cat}]')
    for _, row in group.iterrows():
        print(f'    {row["metric_key"]:<45} "{row["metric"]}"')

# ── Print goalkeeper metrics by category ──────────────────────────
print()
print('=' * 70)
print('GOALKEEPER METRICS')
print('=' * 70)
for cat, group in summary[summary['player_type'] == 'goalkeeper'].groupby('category'):
    print(f'\n  [{cat}]')
    for _, row in group.iterrows():
        print(f'    {row["metric_key"]:<45} "{row["metric"]}"')

Total unique metrics : 67
Outfield metrics     : 55
Goalkeeper metrics   : 43

OUTFIELD PLAYER METRICS

  [attack]
    Offsides                                      "Offsides"
    accurate_crosses                              "Accurate crosses"
    big_chance_missed_title                       "Big chances missed"
    corners                                       "Corners"
    dispossessed                                  "Dispossessed"
    dribbles_succeeded                            "Successful dribbles"
    expected_goals_non_penalty                    "xG Non-penalty"
    long_balls_accurate                           "Accurate long balls"
    passes_into_final_third                       "Passes into final third"
    shots_woodwork                                "Hit woodwork"
    touches                                       "Touches"
    touches_opp_box                               "Touches in opposition box"

  [defense]
    clearance_off_the_line                        "Clear

In [11]:
# Occurrence count = how many times this metric appears across all player-match rows
occurrence = (
    stats_df
    .assign(player_type=stats_df['is_goalkeeper'].map({True: 'goalkeeper', False: 'outfield'}))
    .groupby(['player_type', 'category', 'metric_key', 'metric'])
    .agg(
        total_occurrences = ('value', 'count'),
        matches_appeared  = ('match_id', 'nunique'),
        players_appeared  = ('player_id', 'nunique'),
        pct_non_null      = ('value', lambda x: f"{x.notna().mean()*100:.1f}%"),
        avg_value         = ('value', lambda x: round(x.mean(), 3)),
    )
    .reset_index()
    .sort_values(['player_type', 'category', 'total_occurrences'], ascending=[True, True, False])
    .reset_index(drop=True)
)

# ── Print outfield ─────────────────────────────────────────────────
print('=' * 90)
print('OUTFIELD — metric occurrences')
print('=' * 90)
print(f"{'metric_key':<45} {'category':<18} {'occurrences':>12} {'matches':>8} {'players':>8} {'non_null':>9} {'avg_val':>8}")
print('-' * 90)
for _, row in occurrence[occurrence['player_type'] == 'outfield'].iterrows():
    print(
        f"{row['metric_key']:<45} "
        f"{row['category']:<18} "
        f"{row['total_occurrences']:>12,} "
        f"{row['matches_appeared']:>8,} "
        f"{row['players_appeared']:>8,} "
        f"{row['pct_non_null']:>9} "
        f"{row['avg_value']:>8}"
    )

# ── Print goalkeeper ───────────────────────────────────────────────
print()
print('=' * 90)
print('GOALKEEPER — metric occurrences')
print('=' * 90)
print(f"{'metric_key':<45} {'category':<18} {'occurrences':>12} {'matches':>8} {'players':>8} {'non_null':>9} {'avg_val':>8}")
print('-' * 90)
for _, row in occurrence[occurrence['player_type'] == 'goalkeeper'].iterrows():
    print(
        f"{row['metric_key']:<45} "
        f"{row['category']:<18} "
        f"{row['total_occurrences']:>12,} "
        f"{row['matches_appeared']:>8,} "
        f"{row['players_appeared']:>8,} "
        f"{row['pct_non_null']:>9} "
        f"{row['avg_value']:>8}"
    )

# ── Save to CSV for reference ──────────────────────────────────────
occurrence.to_csv(OUT_DIR / 'metric_reference.csv', index=False)
print(f'\n✓ Saved metric_reference.csv ({len(occurrence)} metrics)')

OUTFIELD — metric occurrences
metric_key                                    category            occurrences  matches  players  non_null  avg_val
------------------------------------------------------------------------------------------
dispossessed                                  attack                    7,426      262      568    100.0%    0.628
touches                                       attack                    7,426      262      568    100.0%   42.354
touches_opp_box                               attack                    7,426      262      568    100.0%    1.821
passes_into_final_third                       attack                    5,736      262      511    100.0%     4.34
long_balls_accurate                           attack                    4,948      262      478    100.0%    1.611
dribbles_succeeded                            attack                    3,738      262      449    100.0%    0.974
expected_goals_non_penalty                    attack                    3,